# Трансформеры

Обратите внимание, что TargetEncoder ожидает на вход не только колонку для преобразования ( city ), но и целевую переменную ( target ). Поскольку значение target зависит от данных, оно не указывается при создании трансформера, а передаётся при его использовании как параметр метода fit() или fit_transform().

Когда вы вызываете preprocessor.fit_transform(), ColumnTransformer вызывает fit_transform() для каждого обработчика, причём только для указанных столбцов. На выходе получается NumPy-массив с преобразованными колонками (в том же порядке, что в transformers ).

In [1]:
import pandas as pd

data = {
    'age': [25, 30, None, 40, 22],
    'salary': [50000, 60000, 55000, 65000, 62000],
    'city': ['Moscow', 'Spb', 'Moscow', 'Kazan', 'Spb'],
    'target': [1, 0, 1, 0, 1]
}
df = pd.DataFrame(data)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder

imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()
target_encoder = TargetEncoder()

# Создаём ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('impute_age', imputer, ['age']),             
        ('scale_salary', scaler, ['salary']),        
        ('encode_city', target_encoder, ['city'])                 
    ]
)
# Обучаем и применяем ColumnTransformer
df_transformed = preprocessor.fit_transform(df, df['target'])

print('Данные до преобразования:')
print(df.head())

print('\nДанные после преобразования:')
print(df_transformed)

Данные до преобразования:
    age  salary    city  target
0  25.0   50000  Moscow       1
1  30.0   60000     Spb       0
2   NaN   55000  Moscow       1
3  40.0   65000   Kazan       0
4  22.0   62000     Spb       1

Данные после преобразования:
[[25.         -1.58069085  0.65674043]
 [30.          0.30108397  0.58581489]
 [29.25       -0.63980344  0.65674043]
 [40.          1.24197138  0.52193492]
 [22.          0.67743894  0.58581489]]


## Pipeline на трансформерах

In [13]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

# 1. Подготовим данные с правильными названиями колонок (X) и целью (y)
data = {
    'температура': [20, 25, 22, 18, 30],
    'влажность': [40, 60, 50, 70, 30],
    'погода': ['ясно', 'облачно', 'дождь', 'ясно', 'облачно'],
    'target': [1, 5, 2, 0, 8] # то, что мы предсказываем
}
df = pd.DataFrame(data)

# Разделяем на признаки и целевую переменную
X = df.drop(columns=['target'])
y = df['target']

# 2. Настраиваем инструменты обработки
scaler = StandardScaler()
one_hot_encoder = OneHotEncoder()

# Собираем ColumnTransformer для обработки признаков
preprocessor = ColumnTransformer(
    transformers=[
        ('num', scaler, ['температура', 'влажность']),
        ('cat', one_hot_encoder, ['погода'])
    ]
)

model = LinearRegression()

# Создаём Pipeline 
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor), # сначала обработка данных
    ('classifier', model)            # затем обучение модели
])

# 4. Обучаем весь пайплайн целиком
# Теперь X — это DataFrame, и в нем есть все нужные колонки
pipeline.fit(X, y)

# 5. Делаем предсказание
predictions = pipeline.predict(X)

print("Первые 3 предсказания:", predictions[:3])
print("\nСхема пайплайна:")
print(pipeline)

Первые 3 предсказания: [1. 5. 2.]

Схема пайплайна:
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['температура', 'влажность']),
                                                 ('cat', OneHotEncoder(),
                                                  ['погода'])])),
                ('classifier', LinearRegression())])


Здесь передаём в пайплайн исходные данные, которые не проходили предобработку. Всё происходит внутри пайплайна: при вызове методов `fit()` или `predict()` сначала запускается обработка признаков, а затем модель получает подготовленные данные.

Задание

Представьте, что вы решаете задачу построения модели для предсказания энергопотребления по погоде. Для этого нужно выполнить следующие шаги:

- К числовым признакам ( температура, влажность ) применить MinMaxScaler.
- К категориальным признакам ( погода, город ) применить TargetEncoder.
- Построить пайплайн ( Pipeline ): сначала обработать признаки через ColumnTransformer, затем обучить модель Ridge.
- Обучить и предсказать значения на обучающей выборке с помощью fit_predict.
- В коде при объявлении ColumnTransformer было пропущено несколько строк. Ваша задача — восстановить их. В этом вам поможет пример из урока.

In [16]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from category_encoders import TargetEncoder


X_train = pd.DataFrame({
    'температура': [16.2, 21.5, 18.0, 25.1, 19.3, 22.8],
    'влажность':   [0.55, 0.40, 0.70, 0.35, 0.60, 0.45],
    'погода':      ['ясно', 'дождь', 'облачно', 'ясно', 'дождь', 'облачно'],
    'город':       ['СПб', 'Москва', 'СПб', 'Казань', 'Москва', 'Казань']
})
y_train = pd.Series([12.0, 8.5, 10.2, 14.3, 9.0, 13.1], name='энергопотребление')

numeric_features = ['температура', 'влажность']
categorical_features = ['погода', 'город']

preprocessor = ColumnTransformer(
            transformers=[
                    ('num_scale', MinMaxScaler(), numeric_features),              
                    ('cat_encode', TargetEncoder(), categorical_features),              
                ]
) # допишите код с созданием ColumnTransformer

pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', Ridge())
])
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_train)
print(predictions)

[11.20673212 10.32116934 10.73215972 12.86387649  9.79326235 12.18279999]
